In [ ]:
import pandas as pd
import numpy as np
import re
import joblib
from pathlib import Path
from datetime import datetime, timedelta

MODEL_DIR = Path("models_12h")

SCALE = 1.018
OFFSET = 1

def parse_metar(metar):
    out = {"vis": np.nan, "wind": 0, "temp": np.nan,
           "dew": np.nan, "fog": 0, "mist": 0}

    m = re.search(r"\b(\d{4})\b", metar)
    if m:
        out["vis"] = int(m.group(1))

    m = re.search(r"(\d{2,3})KT", metar)
    if m:
        out["wind"] = int(m.group(1))

    m = re.search(r"(M?\d{2})/(M?\d{2})", metar)
    if m:
        out["temp"] = int(m.group(1).replace("M", "-"))
        out["dew"] = int(m.group(2).replace("M", "-"))

    out["fog"] = int("FG" in metar)
    out["mist"] = int(("BR" in metar) or ("MIFG" in metar))

    return out

FEATURES = [
    "wind", "temp", "dew", "fog", "mist",
    "hour", "month", "night", "winter"
] + [f"vis_lag_{l}" for l in range(1, 13)]

# Load models
models = {}
for step in range(1, 25):
    path = MODEL_DIR / f"vis_t+{step}.pkl"
    if path.exists():
        models[step] = joblib.load(path)
print(f"Loaded {len(models)} models")


recent_metars = [
    "METAR VIDP 270000Z 25003KT 3000 HZ NSC 29/13 Q1003 NOSIG=",
    "METAR VIDP 270030Z 27003KT 2500 HZ FEW100 29/12 Q1003 NOSIG=",
    "METAR VIDP 270100Z 26003KT 2200 HZ FEW100 29/13 Q1003 NOSIG=",
    "METAR VIDP 270130Z 26005KT 2200 HZ FEW100 29/15 Q1003 NOSIG=",
    "METAR VIDP 270200Z 24005KT 2000 HZ FEW100 30/13 Q1004 NOSIG=",
    "METAR VIDP 270230Z 26006KT 2000 HZ FEW100 32/12 Q1004 NOSIG=",
    "METAR VIDP 270300Z 26006KT 2300 HZ FEW100 33/12 Q1004 NOSIG=",
    "METAR VIDP 270330Z 27004KT 2300 HZ FEW100 34/12 Q1004 NOSIG=",
    "METAR VIDP 270400Z 27004KT 2500 HZ NSC 35/12 Q1004 NOSIG=",
    "METAR VIDP 270430Z 30005KT 2800 HZ NSC 37/12 Q1004 NOSIG=",
    "METAR VIDP 270500Z 32007KT 3500 HZ NSC 38/11 Q1004 NOSIG=",
    "METAR VIDP 270530Z 30006KT 4000 HZ NSC 38/10 Q1004 NOSIG=",
]


latest_valid = datetime(2026, 4, 27, 11, 0)  


latest = parse_metar(recent_metars[-1])

vis_values = [parse_metar(m)["vis"] for m in recent_metars[-12:]]

lag_features = {}
for l in range(1, 13):
    idx = -(l + 1) if l % 3 == 0 else -l
    try:
        val = vis_values[idx]
    except IndexError:
        val = np.nan
    lag_features[f"vis_lag_{l}"] = val * SCALE if not np.isnan(val) else np.nan

hour  = latest_valid.hour
month = latest_valid.month

row = {
    "wind":   latest["wind"],
    "temp":   latest["temp"],
    "dew":    latest["dew"],
    "fog":    latest["fog"],
    "mist":   latest["mist"],
    "hour":   hour,
    "month":  month,
    "night":  int(hour >= 18 or hour <= 6),
    "winter": int(month in [11, 12, 1, 2]),
    **lag_features
}

X = pd.DataFrame([row])[FEATURES]


results = []
for step in range(1, 25):
    pred_vis = models[step].predict(X)[0]
    forecast_time = latest_valid + timedelta(hours=step)
    results.append({
        "forecast_time (IST)": forecast_time.strftime("%Y-%m-%d %H:%M"),
        "step (hrs)": step,
        "predicted_visibility (m)": round(pred_vis)
    })

forecast_df = pd.DataFrame(results)
print(forecast_df.to_string(index=False))

forecast_df.to_excel("vidp_visibility_forecast.xlsx", index=False)
print("\n Forecast saved to vidp_visibility_forecast.xlsx")

Loaded 24 models
forecast_time (IST)  step (hrs)  predicted_visibility (m)
   2026-04-27 12:00           1                      3293
   2026-04-27 13:00           2                      4643
   2026-04-27 14:00           3                      4740
   2026-04-27 15:00           4                      4314
   2026-04-27 16:00           5                      4973
   2026-04-27 17:00           6                      5025
   2026-04-27 18:00           7                      4776
   2026-04-27 19:00           8                      4511
   2026-04-27 20:00           9                      4772
   2026-04-27 21:00          10                      4329
   2026-04-27 22:00          11                      3538
   2026-04-27 23:00          12                      4322
   2026-04-28 00:00          13                      4539
   2026-04-28 01:00          14                      3986
   2026-04-28 02:00          15                      3810
   2026-04-28 03:00          16                      27